# Quantus Focus — smoke test and validation

This notebook builds randomly sampled 2×2 mosaics in which the same target-class image appears twice and the other two images come from classes different from the target. It then computes Focus for the configured explainers, compares the Quantus score with a direct implementation of the formula, and displays the mosaics, target-position masks, and attribution maps.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

while not Path("lib").exists() and Path.cwd().parent != Path.cwd():
    os.chdir("..")

import numpy as np
import pandas as pd
import quantus
import torch

from evaluate_focus import (
    EXPLAINER_NAMES,
    build_focus_mosaics,
    focus_scores_from_attributions,
    format_positions,
    index_dataset_by_class,
    positions_to_masks,
    seed_everything,
)
from lib.defaults import get_default_kwargs
from lib.evaluator import default_explainers
from lib.helpers import plot_example_grid
from lib.setup import setup_notebook
from lib.surrogates import soften_module_inplace_

## Configuration

The smoke test uses Imagenette for speed. Set `DATASET` to `imagenet` before running the full experiment.

In [ ]:
MODEL_NAME = "resnet50"
MODEL_SOURCE = "torchvision"
DATASET = "imagenette"
N_MOSAICS = 4
SEED = 314
USE_ABS = False

seed_everything(SEED)

## Model, data, and mosaics

In [ ]:
device, loader, model = setup_notebook(
    batch_size=1,
    model_name=MODEL_NAME,
    model_source=MODEL_SOURCE,
    seed=SEED,
    dataset=DATASET,
)
temperatures, _, _, _ = get_default_kwargs()
soften_module_inplace_(
    model,
    temperatures=temperatures,
    standard_backward=False,
    fill_default_temperatures=True,
)

dataset = loader.dataset
class_to_indices = index_dataset_by_class(dataset)
mosaics = build_focus_mosaics(
    dataset,
    class_to_indices,
    n_mosaics=N_MOSAICS,
    rng=np.random.default_rng(SEED),
)

print("device:", device)
print("dataset size:", len(dataset))
print("available classes:", len(class_to_indices))
print("mosaics:", mosaics.images.shape)
print("targets:", mosaics.targets.tolist())

## Validate mosaic construction

In `source_indices`, the target image index must appear exactly twice at the positions marked with ones. Labels in the remaining positions must differ from the target.

In [ ]:
metadata_rows = []
for index, (target, positions, component_labels, source_indices) in enumerate(
    zip(
        mosaics.targets,
        mosaics.positions,
        mosaics.component_labels,
        mosaics.source_indices,
    )
):
    target_source_indices = [
        source_index
        for source_index, is_target in zip(source_indices, positions)
        if is_target
    ]
    assert len(target_source_indices) == 2
    assert target_source_indices[0] == target_source_indices[1]
    assert all(
        label == target if is_target else label != target
        for label, is_target in zip(component_labels, positions)
    )
    metadata_rows.append(
        {
            "mosaic": index,
            "target": int(target),
            "target_positions": format_positions(positions),
            "component_labels": tuple(component_labels),
            "source_indices": tuple(source_indices),
        }
    )

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)

In [ ]:
plot_example_grid(
    torch.from_numpy(mosaics.images),
    nrow=2,
    title="Focus mosaics",
)

target_masks = positions_to_masks(mosaics.positions, mosaics.images.shape)
plot_example_grid(
    torch.from_numpy(target_masks),
    nrow=2,
    normalize=False,
    value_range=(0, 1),
    cmap="gray",
    title="Target quadrants (white)",
)

## Attributions and Focus

In [ ]:
explainers = default_explainers(EXPLAINER_NAMES)
attributions = {}

for explainer_name, (explain_func, explain_kwargs) in explainers.items():
    print(f"Computing {explainer_name}...")
    attributions[explainer_name] = explain_func(
        model,
        mosaics.images,
        mosaics.targets,
        device=device,
        **explain_kwargs,
    )
    print(explainer_name, attributions[explainer_name].shape)

In [ ]:
score_rows = []
for explainer_name, attrs in attributions.items():
    metric = quantus.Focus(
        abs=USE_ABS,
        normalise=False,
        return_aggregate=False,
        disable_warnings=True,
    )
    quantus_scores = np.asarray(
        metric(
            model=model,
            x_batch=mosaics.images,
            y_batch=mosaics.targets,
            a_batch=attrs,
            custom_batch=mosaics.positions,
            channel_first=True,
            device=device,
        )
    )
    manual_scores = focus_scores_from_attributions(
        attrs, mosaics.positions, use_abs=USE_ABS
    )
    np.testing.assert_allclose(quantus_scores, manual_scores, atol=1e-6)

    for mosaic_index, score in enumerate(quantus_scores):
        score_rows.append(
            {
                "explainer": explainer_name,
                "mosaic": mosaic_index,
                "target": int(mosaics.targets[mosaic_index]),
                "positions": format_positions(mosaics.positions[mosaic_index]),
                "focus": float(score),
            }
        )

scores_df = pd.DataFrame(score_rows)
display(scores_df)
display(scores_df.groupby("explainer")["focus"].agg(["mean", "std", "count"]))

## Attribution visualisation

A good explanation should concentrate positive relevance in the two target quadrants shown in white above.

In [ ]:
for explainer_name, attrs in attributions.items():
    plot_example_grid(
        torch.from_numpy(attrs),
        nrow=2,
        heatmap=True,
        heatmap_mode="mean",
        title=f"{explainer_name} attributions",
    )